# 10 · Changing data: INSERT, UPDATE, MERGE, SCD Type 2, incremental loads (Oracle)

**DML** (Data Manipulation Language) changes rows: `INSERT`, `UPDATE`, `DELETE`, `MERGE`.
**DDL** (Data Definition Language) changes tables: `CREATE`, `ALTER`, `DROP`.

This notebook works on **copies** named `prac_*`, so the 7 practice tables never change.
Each question has a setup cell that rebuilds its copy: run it first, and again to start over.

Data engineering words used here:
- **upsert** = update the row if the key exists, insert it if not (`MERGE`)
- **SCD Type 2** (slowly changing dimension) = keep the **history**: close the old row, add a new current row
- **incremental load** = load only the new rows since last time (using a **watermark**, e.g. the max date already loaded)
- **idempotent** = running the job twice gives the same result as running it once

**How to run:** Start each SQL cell with `%%oracle`. One statement per cell. Tables are in your own schema, so just write `employees`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "oracle"))
%load_ext oracle_helper

## Q1. INSERT a row  ·  🟢 easy

Add customer **41**, `Test User`, city `Austin`, signup `2024-12-01` to `prac_customers`. Then show `num_customers` (the row count).

*Expected result: 1 row · columns: `num_customers`*

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

In [ ]:
%%oracle
-- your answer here

## Q2. UPDATE with a condition  ·  🟢 easy

In `prac_customers`, set `city` to `Unknown` where it is NULL. Then show `customer_id`, `name`, `city` of the changed rows.

*Expected result: 1 row · columns: `customer_id`, `name`, `city`*

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

In [ ]:
%%oracle
-- your answer here

## Q3. DELETE with NOT EXISTS  ·  🟢 easy

Delete from `prac_customers` everyone who **never ordered**. Then show `num_customers`.

*Expected result: 1 row · columns: `num_customers`*

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

In [ ]:
%%oracle
-- your answer here

## Q4. Upsert with MERGE  ·  🟡 medium

`prac_customer_updates` has 4 rows: customers 1 and 2 changed, 41 and 42 are new. MERGE them into `prac_customers`: update `name` and `city` when the id exists, insert when not. Then show customers 1, 2, 41, 42.

*Expected result: 4 rows · columns: `customer_id`, `name`, `city`, `signup_date`*

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customer_updates PURGE

In [ ]:
%%oracle
CREATE TABLE prac_customer_updates (
    customer_id NUMBER(10), name VARCHAR2(100), city VARCHAR2(50), signup_date DATE)

In [ ]:
%%oracle
INSERT INTO prac_customer_updates
SELECT 1,  'Aarav Nguyen',    'Denver',  DATE '2023-01-26' FROM dual UNION ALL
SELECT 2,  'Priya Silva-Rao', 'Seattle', DATE '2023-09-08' FROM dual UNION ALL
SELECT 41, 'Nina Patel',      'Boston',  DATE '2024-12-05' FROM dual UNION ALL
SELECT 42, 'Omar Haddad',     'Austin',  DATE '2024-12-06' FROM dual

In [ ]:
%%oracle
-- your answer here

## Q5. Idempotent reload of one slice  ·  🟡 medium

`prac_orders` is a raw copy of `orders` (with duplicates). Reload **October 2025** from a clean source (`SELECT DISTINCT * FROM orders`): first DELETE the slice, then INSERT it again. Run both cells twice: the result must not change. Show `oct_rows` and `oct_orders` for October 2025.

*Expected result: 1 row · columns: `oct_rows`, `oct_orders`*

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_orders PURGE

In [ ]:
%%oracle
CREATE TABLE prac_orders AS
SELECT * FROM orders

In [ ]:
%%oracle
-- your answer here

## Q6. Incremental load with a watermark  ·  🟡 medium

`prac_orders_target` holds clean orders up to 2025-06-30. Load only orders **newer than the latest date already in the target** (the watermark). Then show `num_rows` and `max_date` of the target.

*Expected result: 1 row · columns: `num_rows`, `max_date`*

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_orders_target PURGE

In [ ]:
%%oracle
CREATE TABLE prac_orders_target AS
SELECT DISTINCT * FROM orders WHERE order_date <= DATE '2025-06-30'

In [ ]:
%%oracle
-- your answer here

## Q7. SCD Type 2: keep the history  ·  🔴 hard

`prac_dim_customer` has one current row per customer (`valid_to = 9999-12-31`, `is_current = 'Y'`). `prac_city_changes` says: customer 1 → Denver, customer 3 → Phoenix (no change!), customer 13 → Austin (old city is NULL). Changes start on **2025-01-15**.

1. **Close** the current row of customers whose city really changed: `valid_to = 2025-01-14`, `is_current = 'N'`.
2. **Insert** a new current row for them from 2025-01-15.
3. Show the rows of customers 1, 3, 13.

*Expected result: 5 rows · columns: `customer_id`, `city`, `valid_from`, `valid_to`, `is_current`*

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_dim_customer PURGE

In [ ]:
%%oracle
CREATE TABLE prac_dim_customer AS
SELECT customer_id, city, signup_date AS valid_from,
       DATE '9999-12-31' AS valid_to, CAST('Y' AS VARCHAR2(1)) AS is_current
FROM customers

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_city_changes PURGE

In [ ]:
%%oracle
CREATE TABLE prac_city_changes (customer_id NUMBER(10), city VARCHAR2(50))

In [ ]:
%%oracle
INSERT INTO prac_city_changes
SELECT 1, 'Denver' FROM dual UNION ALL
SELECT 3, 'Phoenix' FROM dual UNION ALL      -- same city as now: NOT a change
SELECT 13, 'Austin' FROM dual                -- old city is NULL

In [ ]:
%%oracle
-- your answer here

## Q8. ALTER TABLE: add a flag column  ·  🟡 medium

Add a column `is_active` (1 letter) to `prac_customers`. Set it to `Y` for customers with at least one order, `N` for the rest. Show `is_active` and `num_customers`.

*Expected result: 2 rows · columns: `is_active`, `num_customers`*

In [ ]:
%%oracle
DROP TABLE IF EXISTS prac_customers PURGE

In [ ]:
%%oracle
CREATE TABLE prac_customers AS
SELECT * FROM customers

In [ ]:
%%oracle
-- your answer here